# Testing *"When a linear-predictor LeJEPA separates dynamic regimes"* with learnt encoders

The proposal derives, for encoders that are **linear on a fixed feature map Φ** with **exact whitening** and the
**population loss**, when a JEPA with a linear predictor keeps regime information and how a *shared* encoder differs
from *separate* context/target encoders. This notebook asks whether those predictions survive the setting that matters
in practice:

| proposal assumes | here |
|---|---|
| Φ fixed, encoder linear on it | Φ **learnt**: a TCN (default), GRU or Transformer backbone + le-wm style projector |
| exact whitening | **SIGReg**, verbatim from the le-wm repo (LeJEPA's regulariser), one penalty per encoder; exact batch whitening as a control |
| population loss | finite data, minibatch AdamW |
| the state and regime are observed through Φ | each JEPA step is a **window of 16 samples × 24 channels**: the latent variables are up-projected, passed through a random nonlinear warp, given per-channel gain/offset, intra-window jitter, Gaussian noise and Student-t outliers |

The data-generating process is controlled exactly (regime chain $p_{kj}$, regime dynamics $B_k$, carried or redrawn
state, regime-free competitors), so every claim can be turned into a **numerical prediction** before training.

### How predictions are made: the oracle
For each dataset the notebook builds $\Phi_{\text{oracle}}$ = all monomials of degree ≤ 3 of the true state and nuisance
factors, each multiplied by every regime indicator, plus the indicators. This is a Φ that "contains the features used"
in the proposal. On a long simulation it whitens Φ, estimates $K=\mathbb E[\Phi_{t+1}\Phi_t^\top]$ and computes
* **separate** optimum: top-$d$ singular vectors of $K$, score $\sum_{j\le d}\sigma_j^2$;
* **shared** optimum: $\max_{U^\top U=I}\|U^\top K U\|_F^2$ by Riemannian ascent (with the §5 bracket printed).

The oracle latents are evaluated with **exactly the same metrics** as the learnt latents, so each table shows
*theory* rows next to *learnt* rows.

### Metrics
* **score** — predictable variance $\|C\|_F^2$ with $C=\mathbb E[Y_{t+1}Z_t^\top]$ after exact empirical whitening of each encoder's outputs (the quantity the proposal says training maximises; $\min_A\mathcal L=d-\text{score}$).
* **R²Z f / R²Y f** — linear $R^2$ of ground-truth feature $f$ from the context latent $Z$ / target latent $Y$: which features were kept. Features: regime indicator, $x$, $g_k=x\,\mathbf 1[S=k]/\sqrt{\pi_k}$, $|x|^2$, nuisance $n$, $n^2$.
* **regime readouts** (balanced accuracy, chance $1/K$): linear and MLP probes on $Z$ (and $Y$, $[Z,Y]$); unsupervised GMM on $Z$ (*mean separation*); for separate encoders, $\operatorname{sign}(z\cdot y)$ and an unsupervised **switching linear regression** between $Z$ and $Y$ computed on the *same* input window (§6).

### Claims under test
| id | claim (section of the proposal) | experiment |
|---|---|---|
| C1 | conflicting dynamics make regime-distinguishing features valuable, but only if they are among the most predictable; $x^2$ can win and the regime is discarded (§4, §7, concl. 1) | E2 |
| C2 | separate ≥ shared in predictable variance; rotation: shared $d{=}1$ gets 0 with a linear Φ, separate $r^2$; equal at $d{=}2$ (§5, concl. 2) | E1 |
| C3 | carried state: shared regime directions scale with $p$, the separate pair gets $\rho^2$ whatever $p$; redrawing the state brings $p_{kk}$ back (§6, concl. 3) | E2, E2b |
| C4 | separate encoders can hold the regime in *neither* latent alone; it is the sign of $z\cdot y$ and a switching regression recovers it (§6, concl. 4) | E2, E5c |
| C5 | mean separation depends only on the regime chain; equal in both designs without a preferred order; a preferred (cyclic) order favours separate encoders (§6, concl. 5) | E2, E3 |
| C6 | regime-free features compete for the $d$ dimensions; onset of the regime predicted by counting Hermite polynomials (§7, concl. 6) | E4 |
| C7 | SIGReg is a soft, Gaussian-shaped stand-in for whitening; it opposes two-valued regime coordinates, less so at large $d$ (§8) | E5b |
| C6′ | the same competition with slow features, three conflicting regimes and $d$ up to 32 | E6 |
| C8 | the "missing tests" of §9: nonlinear predictor, readout robustness on trained encoders, backbone dependence | E5a, E5c, E5d |

Each experiment ends with automatic `check(...)` lines comparing the learnt models with the prediction; Section 6
collects them in a scoreboard. **A "NOT consistent" line is a result, not a bug**: the point is to find where the
idealised theory stops describing learnt encoders.

Runtime: ≈ 70 trainings of 1500 steps on an Apple-silicon GPU (MPS) ≈ 1–1.5 h; finished runs are cached in
`results_cache.pkl`. Set `FAST = True` for a smoke run.

## 0. Setup

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(_v, "4")      # small matrices: many threads only oversubscribe the CPU
import math, time, itertools, warnings
from dataclasses import dataclass, field, replace
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
torch.set_num_threads(4)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print('device:', DEVICE)

In [ ]:
# ---- run configuration ------------------------------------------------------------------------
FAST = False            # True: fewer steps / smaller data for a quick smoke run (~10 min)
STEPS = 600 if FAST else 1500
SEEDS = [0] if FAST else [0, 1]          # seeds for the headline experiments (E2, E3)
SERIES = (16, 8) if FAST else (48, 16)   # (train, test) series of 1000 windows each
ORACLE = dict(n_series=40, T=2500)       # length of the simulation the theory oracle is computed on
CACHE = "results_cache.pkl"              # finished runs are cached; delete the file to recompute
import os, pickle
import matplotlib.pyplot as plt

_cache = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}
_datasets = {}

def get_data(spec):
    key = repr(spec)
    if key not in _datasets:
        _datasets[key] = make_dataset(spec, n_train=SERIES[0], n_test=SERIES[1])
    return _datasets[key]

READOUT_VERSION = 2                      # bump after changing a readout: metrics are recomputed from saved latents
LATENTS = "latents"                      # test-set latents of every trained model
os.makedirs(LATENTS, exist_ok=True)
import hashlib, json

def cached_run(spec, design, d, tag="", **kw):
    kw.setdefault("steps", STEPS)
    for k, v in dict(backbone="tcn", predictor="linear", reg="sigreg", lam=0.1).items():
        kw.setdefault(k, v)
    key = (repr(spec), design, d, tuple(sorted(kw.items())))
    fname = f"{LATENTS}/{hashlib.md5(repr(key).encode()).hexdigest()}.npz"
    ent = _cache.get(key)
    if ent is None or ent.get("readout_version") != READOUT_VERSION:
        t0 = time.time()
        if os.path.exists(fname):
            f = np.load(fname); Z, Y, info = f["Z"], f["Y"], json.loads(str(f["info"]))
        else:
            Z, Y, info = train_encode(get_data(spec), design, d, **kw)
            np.savez_compressed(fname, Z=Z, Y=Y, info=json.dumps(info))
        res = evaluate_latents(Z.astype(np.float64), Y.astype(np.float64), spec, get_data(spec)["test"], design == "shared")
        ent = dict(design=design, d=d, **{k: kw[k] for k in ("backbone", "predictor", "reg", "lam", "seed") if k in kw},
                   **res, **info, readout_version=READOUT_VERSION, secs=round(time.time() - t0, 1))
        _cache[key] = ent
        pickle.dump(_cache, open(CACHE, "wb"))
        with open("progress.log", "a") as f:   # follow with: tail -f progress.log
            f.write(f"{time.strftime('%H:%M:%S')}  run {len(_cache):3d}  {tag:<12} {spec.name:<28} {design:<8} d={d:<3} "
                    f"score={ent['score']:.3f}  secs={ent['secs']}\n")
    r = dict(ent); r["tag"] = tag
    return r

CLU = ["clu GMM", "clu spectral kNN", "clu subspace (TSC)", "clu HMM", "clu AR-HMM (z_t→z_t+1)", "clu HMM switch-reg (Z→Y)"]

def oracle(spec, d_list, **kw):
    return oracle_analysis(spec, d_list, **{**ORACLE, **kw})

CHECKS = []
def check(claim, statement, ok, detail=""):
    CHECKS.append(dict(claim=claim, prediction=statement, consistent="yes" if ok else "NO", observed=detail))
    print(f"[{'consistent' if ok else 'NOT consistent'}] {claim}: {statement}  |  {detail}")

def show(df, cols):
    cols = [c for c in cols if c in df.columns]
    display(df[cols].round(3))

def mean_over_seeds(df, keys):
    num = df.select_dtypes("number").columns.difference(keys + ["seed", "secs"])
    return df.groupby(keys, sort=False)[list(num)].mean().reset_index()

C_SH, C_SEP, C_TH = "#2a78d6", "#eb6834", "#52514e"   # shared, separate, theory
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "lines.linewidth": 2, "figure.dpi": 110, "font.size": 9})
R90 = ((0.0, -1.0), (1.0, 0.0))
def two(p):
    return ((p, 1 - p), (1 - p, p))

## 1. SIGReg (from le-wm) and exact whitening

`SIGReg` below is copied verbatim from [`lucas-maes/le-wm/module.py`](https://github.com/lucas-maes/le-wm/blob/main/module.py):
for 1024 random unit directions it computes the Epps–Pulley statistic between the projected batch and $\mathcal N(0,1)$
on 17 knots of the characteristic function. It is called with a leading singleton time axis, `(1, B, d)`.

Following §8, there is **one penalty per encoder**: the shared design penalises the pooled outputs of its single encoder;
the separate design penalises each encoder's outputs (averaged). Training follows LeJEPA/LeWM: no EMA and no
stop-gradient, loss $=\text{MSE}(Y_{t+1},AZ_t)+\lambda\,\text{SIGReg}$ with $\lambda=0.1$ (LeWM uses 0.09).
`batch_whiten` (Cholesky whitening of each batch) is the exact-whitening control used in E5b.

In [ ]:
class SIGReg(torch.nn.Module):
    """Sketch Isotropic Gaussian Regularizer (single-GPU!)

    Verbatim from lucas-maes/le-wm, module.py (the LeWM implementation of LeJEPA's SIGReg):
    Epps-Pulley test of every random 1-D projection against N(0,1), on 17 knots in [0,3].
    """

    def __init__(self, knots=17, num_proj=1024):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt
        window = torch.exp(-t.square() / 2.0)
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj):
        """
        proj: (T, B, D)
        """
        # sample random projections
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        # compute the epps-pulley statistic
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        statistic = (err @ self.weights) * proj.size(-2)
        return statistic.mean()  # average over projections and time


def batch_whiten(z, eps=1e-4):
    """Exact (in-batch) Cholesky whitening: the idealised constraint of the theory (W-MSE / VAMPnet style)."""
    zc = z - z.mean(0, keepdim=True)
    cov = zc.T @ zc / (z.shape[0] - 1) + eps * torch.eye(z.shape[1], device=z.device)
    L = torch.linalg.cholesky(cov.cpu()).to(z.device)  # cholesky is not implemented on MPS
    return torch.linalg.solve_triangular(L, zc.T, upper=False).T

## 2. Realistic synthetic data with a controlled generator

Window-level ground truth (one JEPA step = one window):
* regime $S_t$: Markov chain with matrix $p_{kj}$, switching independently of the state;
* state $x_t\in\mathbb R^m$: $x_{t+1}=B_{S_t}x_t+\sqrt{1-\rho_{S_t}^2}\,\varepsilon$, with $B_k=\rho_kQ_k$ ($Q_k$ orthogonal), so $\operatorname{cov}(x)=I$ in every regime ($M_k=I$); the state is **carried** across switches unless `redraw=True`;
* nuisance $n_t\in\mathbb R^q$: independent AR(1), autocorrelation $\lambda$ — regime-free competitors (§7).

Rendering of window $t$ into 16 samples × 24 channels: $u_s=[x_t+\text{jitter},\ n_t+\text{jitter},\ a\cos(2\pi f_{S_t}s/16+\phi_t)]$
(the cosine with random phase is the regime cue that makes the regime *visible* to the encoder, as the proposal
assumes — its mean is zero, so only a nonlinear Φ reads it), then $o=Pu$ (up-projection), $o\leftarrow o+\beta\,W_2\tanh(W_1o)$ (warp),
per-channel gain/offset, Gaussian noise and 2 % Student-t(3) outliers, then channel standardisation.
Consecutive windows do not overlap, as assumed.

In [ ]:
@dataclass
class DataSpec:
    name: str = "worked"
    P: tuple = ((0.9, 0.1), (0.1, 0.9))   # regime transition matrix p_kj
    rho: tuple = (0.8, -0.8)              # B_k = rho_k Q_k
    Q: tuple = None                       # orthogonal m x m per regime (default identity)
    m: int = 1                            # state dimension
    redraw: bool = False                  # redraw the state at a switch instead of carrying it
    q: int = 0                            # nuisance AR(1) factors (regime-free competitors)
    lam: float = 0.9                      # their autocorrelation
    cue: float = 1.0                      # amplitude of the regime cue in the observations (0 = none)
    W: int = 16                           # samples per window (one JEPA step = one window)
    D: int = 24                           # observed channels (up-projection)
    warp: float = 1.0                     # strength of the random nonlinear mixing
    obs_noise: float = 0.3                # additive observation noise
    jitter: float = 0.15                  # intra-window jitter of the latent variables
    heavy: float = 0.02                   # fraction of samples hit by Student-t(3) outliers
    render_seed: int = 0                  # fixes the (unknown to the model) emission map

    @property
    def K(self):
        return len(self.P)

    @property
    def pi(self):
        P = np.asarray(self.P, float)
        w, v = np.linalg.eig(P.T)
        p = np.real(v[:, np.argmin(abs(w - 1))])
        return p / p.sum()


def simulate_latents(spec, n_series, T, rng):
    """Ground-truth window-level process: regime S_t (Markov), state x_t carried across switches, nuisance n_t."""
    P = np.asarray(spec.P, float)
    K, m, q = spec.K, spec.m, spec.q
    Q = [np.eye(m)] * K if spec.Q is None else [np.asarray(Qk, float) for Qk in spec.Q]
    B = [spec.rho[k] * Q[k] for k in range(K)]
    sd = [math.sqrt(max(1 - spec.rho[k] ** 2, 0.0)) for k in range(K)]  # keeps cov(x)=I in every regime
    S = np.zeros((n_series, T), int)
    x = np.zeros((n_series, T, m))
    n = np.zeros((n_series, T, q))
    S[:, 0] = rng.choice(K, size=n_series, p=spec.pi)
    x[:, 0] = rng.standard_normal((n_series, m))
    n[:, 0] = rng.standard_normal((n_series, q))
    cum = P.cumsum(1)
    for t in range(1, T):
        s = S[:, t - 1]
        u = rng.random(n_series)
        S[:, t] = (u[:, None] > cum[s]).sum(1)
        eps = rng.standard_normal((n_series, m))
        xn = np.einsum("nij,nj->ni", np.stack(B)[s], x[:, t - 1]) + np.array(sd)[s][:, None] * eps
        if spec.redraw:
            sw = S[:, t] != s
            xn[sw] = rng.standard_normal((sw.sum(), m))
        x[:, t] = xn
        n[:, t] = spec.lam * n[:, t - 1] + math.sqrt(1 - spec.lam ** 2) * rng.standard_normal((n_series, q))
    return dict(S=S, x=x, n=n)


def render(spec, lat, rng):
    """Observation windows: up-projection + random nonlinear warp + per-channel gain/offset + noise + outliers."""
    S, x, n = lat["S"], lat["x"], lat["n"]
    N, T = S.shape
    W, D = spec.W, spec.D
    r = np.random.default_rng(1000 + spec.render_seed)
    freqs = np.array([2.0, 3.5, 5.0, 6.5])[: spec.K]
    dim_u = spec.m + spec.q + 1
    P_up = r.standard_normal((D, dim_u)) / math.sqrt(dim_u)
    W1 = r.standard_normal((D, D)) * 1.5 / math.sqrt(D)
    W2 = r.standard_normal((D, D)) / math.sqrt(D)
    gain, offset = r.uniform(0.5, 2.0, D), r.standard_normal(D)
    s = np.arange(W)
    phase = rng.uniform(0, 2 * np.pi, (N, T, 1))
    cue = spec.cue * np.cos(2 * np.pi * freqs[S][..., None] * s / W + phase)            # (N,T,W)
    xs = x[:, :, None, :] + spec.jitter * rng.standard_normal((N, T, W, spec.m))
    ns = n[:, :, None, :] + spec.jitter * rng.standard_normal((N, T, W, spec.q))
    u = np.concatenate([xs, ns, cue[..., None]], -1)                                    # (N,T,W,dim_u)
    o = u @ P_up.T
    o = o + spec.warp * np.tanh(o @ W1.T) @ W2.T
    o = o * gain + offset
    noise = rng.standard_normal(o.shape)
    if spec.heavy > 0:
        hit = rng.random(o.shape) < spec.heavy
        noise[hit] = rng.standard_t(3, hit.sum()) * 3
    o = o + spec.obs_noise * noise
    return o.astype(np.float32)


def make_dataset(spec, n_train=48, n_test=16, T=1000, seed=0):
    rng = np.random.default_rng(seed)
    tr, te = simulate_latents(spec, n_train, T, rng), simulate_latents(spec, n_test, T, rng)
    otr, ote = render(spec, tr, rng), render(spec, te, rng)
    mu, sd = otr.mean((0, 1, 2)), otr.std((0, 1, 2)) + 1e-6
    return dict(spec=spec, train=tr, test=te,
                obs_train=torch.tensor((otr - mu) / sd), obs_test=torch.tensor((ote - mu) / sd))

In [ ]:
spec_demo = DataSpec(P=two(0.9))
demo = make_dataset(spec_demo, n_train=2, n_test=1, T=300)
lat, obs = demo["test"], demo["obs_test"][0].numpy()
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.4, 3]))
t = np.arange(120)
ax[0].step(t, lat["S"][0, :120], where="post", color=C_TH); ax[0].set_ylabel("regime $S_t$"); ax[0].set_yticks([0, 1])
ax[1].plot(t, lat["x"][0, :120, 0], color=C_SH); ax[1].set_ylabel("state $x_t$")
ax[2].imshow(obs[:120].reshape(120 * spec_demo.W, spec_demo.D).T, aspect="auto", cmap="Greys",
             extent=[0, 120, spec_demo.D, 0], interpolation="nearest")
ax[2].set_ylabel("observed channel"); ax[2].set_xlabel("window t (each window = 16 samples × 24 channels)")
ax[2].grid(False)
fig.suptitle("Worked example (p = 0.9, ρ = 0.8) as the encoder sees it: regime cue + state, up-projected, warped, noisy")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3.5))
x0, x1, s0 = lat["x"][0, :-1, 0], lat["x"][0, 1:, 0], lat["S"][0, :-1]
ax.scatter(x0[s0 == 0], x1[s0 == 0], s=9, color=C_SH, label="regime 1 (+ρ)")
ax.scatter(x0[s0 == 1], x1[s0 == 1], s=9, color=C_SEP, label="regime 2 (−ρ)")
ax.set_xlabel("$x_t$"); ax.set_ylabel("$x_{t+1}$"); ax.legend(frameon=False)
ax.set_title("Pooled over regimes the best line is flat (§4)")
plt.tight_layout(); plt.show()

## 3. Encoders, linear-predictor JEPA and training

In [ ]:
class TCN(nn.Module):
    def __init__(self, D, H=64, levels=4, k=3):
        super().__init__()
        self.inp = nn.Conv1d(D, H, 1)
        self.blocks = nn.ModuleList()
        for i in range(levels):
            dil = 2 ** i
            self.blocks.append(nn.Sequential(
                nn.Conv1d(H, H, k, dilation=dil, padding=dil * (k - 1) // 2), nn.GELU(),
                nn.Conv1d(H, H, k, dilation=dil, padding=dil * (k - 1) // 2)))
        self.out_dim = 2 * H

    def forward(self, x):                      # x: (B, W, D)
        h = self.inp(x.transpose(1, 2))
        for b in self.blocks:
            h = h + b(F.gelu(h))
        h = F.gelu(h)
        return torch.cat([h.mean(-1), h.amax(-1)], -1)


class GRUEnc(nn.Module):
    def __init__(self, D, H=64):
        super().__init__()
        self.rnn = nn.GRU(D, H, num_layers=2, batch_first=True)
        self.out_dim = H

    def forward(self, x):
        _, h = self.rnn(x)
        return h[-1]


class TransformerEnc(nn.Module):
    def __init__(self, D, H=64, W=16, layers=2):
        super().__init__()
        self.inp = nn.Linear(D, H)
        self.pos = nn.Parameter(0.02 * torch.randn(1, W, H))
        layer = nn.TransformerEncoderLayer(H, 4, 2 * H, dropout=0.0, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.out_dim = H

    def forward(self, x):
        return self.tf(self.inp(x) + self.pos[:, : x.shape[1]]).mean(1)


BACKBONES = dict(tcn=TCN, gru=GRUEnc, transformer=TransformerEnc)


class Encoder(nn.Module):
    """Learnt feature extractor Phi (backbone) + le-wm style projector (MLP with BatchNorm) -> latent in R^d."""

    def __init__(self, backbone, D, W, d, H=64):
        super().__init__()
        self.phi = BACKBONES[backbone](D, H) if backbone != "transformer" else TransformerEnc(D, H, W)
        h = self.phi.out_dim
        self.proj = nn.Sequential(nn.Linear(h, 2 * H), nn.BatchNorm1d(2 * H), nn.GELU(), nn.Linear(2 * H, d))

    def forward(self, x):
        return self.proj(self.phi(x))


class LinJEPA(nn.Module):
    def __init__(self, design="shared", d=1, backbone="tcn", predictor="linear", reg="sigreg",
                 lam=0.1, D=24, W=16, H=64):
        super().__init__()
        self.design, self.reg, self.lam = design, reg, lam
        self.ctx = Encoder(backbone, D, W, d, H)
        self.tgt = self.ctx if design == "shared" else Encoder(backbone, D, W, d, H)
        self.pred = nn.Linear(d, d, bias=False) if predictor == "linear" else nn.Sequential(
            nn.Linear(d, 128), nn.GELU(), nn.Linear(128, 128), nn.GELU(), nn.Linear(128, d))
        self.sigreg = SIGReg()

    def losses(self, xc, xn):
        if self.design == "shared":           # one encoder, one penalty on its outputs
            out = self.ctx(torch.cat([xc, xn]))
            if self.reg == "whiten":
                out = batch_whiten(out)
            z, y = out.chunk(2)
            reg = self.sigreg(out[None]) if self.reg == "sigreg" else out.new_zeros(())
        else:                                 # two encoders, one penalty each
            z, y = self.ctx(xc), self.tgt(xn)
            if self.reg == "whiten":
                z, y = batch_whiten(z), batch_whiten(y)
                reg = z.new_zeros(())
            else:
                reg = 0.5 * (self.sigreg(z[None]) + self.sigreg(y[None]))
        pred = (y - self.pred(z)).square().mean()   # no stop-gradient, no EMA (LeJEPA / LeWM)
        return pred, reg


def train_jepa(data, design="shared", d=1, backbone="tcn", predictor="linear", reg="sigreg", lam=0.1,
               steps=1500, batch=512, lr=2e-3, seed=0, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    spec = data["spec"]
    model = LinJEPA(design, d, backbone, predictor, reg, lam, spec.D, spec.W).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    obs = data["obs_train"].to(DEVICE)
    N, T = obs.shape[:2]
    g = torch.Generator(device="cpu").manual_seed(seed)
    hist = []
    model.train()
    for step in range(steps):
        i = torch.randint(N, (batch,), generator=g)
        t = torch.randint(T - 1, (batch,), generator=g)
        xc, xn = obs[i, t], obs[i, t + 1]
        pred, rg = model.losses(xc, xn)
        loss = pred + lam * rg
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); sched.step()
        if step % 100 == 0 or step == steps - 1:
            hist.append((step, pred.item(), rg.item()))
            if verbose:
                print(f"{step:5d} pred {pred.item():.4f} reg {rg.item():.4f}")
    model.eval()
    return model, hist


@torch.no_grad()
def encode(model, obs, bs=4096):
    N, T = obs.shape[:2]
    flat = obs.reshape(N * T, *obs.shape[2:])
    zs, ys = [], []
    for i in range(0, len(flat), bs):
        xb = flat[i: i + bs].to(DEVICE)
        zs.append(model.ctx(xb).cpu()); ys.append(model.tgt(xb).cpu())
    Z = torch.cat(zs).reshape(N, T, -1).numpy().astype(np.float64)
    Y = torch.cat(ys).reshape(N, T, -1).numpy().astype(np.float64)
    return Z, Y

## 4. Metrics and the theory oracle

In [ ]:
def predictable_variance(Z, Y, shared):
    """||C||_F^2 with C = E[Y_{t+1} Z_t^T] after exact (empirical) whitening of each encoder's outputs.
    For the shared design the same whitening map is used for both sides (it is one encoder)."""
    d = Z.shape[-1]
    if shared:
        Wd = whiten_map(Z.reshape(-1, d))
        Zw = Yw = (Z - Z.reshape(-1, d).mean(0)) @ Wd
    else:
        Zw = (Z - Z.reshape(-1, d).mean(0)) @ whiten_map(Z.reshape(-1, d))
        Yw = (Y - Y.reshape(-1, d).mean(0)) @ whiten_map(Y.reshape(-1, d))
    a, b = Zw[:, :-1].reshape(-1, Zw.shape[-1]), Yw[:, 1:].reshape(-1, Yw.shape[-1])
    C = b.T @ a / len(a)
    return float((C ** 2).sum()), np.linalg.svd(C, compute_uv=False)


def whiten_map(A, eps=1e-6):
    A = A - A.mean(0)
    C = A.T @ A / len(A)
    w, V = np.linalg.eigh(C)
    w = np.maximum(w, eps * max(w.max(), eps))
    return V / np.sqrt(w)


def gt_features(spec, lat):
    """Named ground-truth features per window, each (N,T,k). These are the candidate features of the theory."""
    S, x, n = lat["S"], lat["x"], lat["n"]
    K, pi = spec.K, spec.pi
    onehot = np.eye(K)[S]
    f = {"regime": onehot[..., : K - 1]} if K > 1 else {}
    f["x"] = x
    f["g (x·1[S=k])"] = np.concatenate([x * onehot[..., k: k + 1] / math.sqrt(pi[k]) for k in range(K)], -1)
    f["|x|²"] = (x ** 2).sum(-1, keepdims=True)
    if spec.m == 2:
        f["x quad. (x1²-x2², x1x2)"] = np.stack([x[..., 0] ** 2 - x[..., 1] ** 2, x[..., 0] * x[..., 1]], -1)
    if spec.q:
        f["nuisance n"] = n
        quad = [n[..., i] * n[..., j] - (i == j) for i in range(spec.q) for j in range(i, spec.q)]
        f["nuisance n²"] = np.stack(quad, -1)
    return f


def r2_table(Lat, feats, n_fit):
    """Linear R^2 of each ground-truth feature from the latent (fit on first n_fit windows, test on the rest)."""
    out = {}
    L = Lat.reshape(-1, Lat.shape[-1])
    for name, F_ in feats.items():
        Fm = F_.reshape(-1, F_.shape[-1])
        r2s = []
        for j in range(Fm.shape[1]):
            reg = LinearRegression().fit(L[:n_fit], Fm[:n_fit, j])
            res = Fm[n_fit:, j] - reg.predict(L[n_fit:])
            r2s.append(1 - res.var() / (Fm[n_fit:, j].var() + 1e-12))
        out[name] = float(np.clip(np.mean(r2s), 0, 1))
    return out


def best_perm_acc(true, pred, K):
    M = np.zeros((K, K))
    for a, b in zip(true, pred):
        M[a, b] += 1
    r, c = linear_sum_assignment(-M)
    mapping = dict(zip(c, r))
    return balanced_accuracy_score(true, [mapping.get(p, -1) for p in pred])


def switching_regression(X, Y, K, restarts=5, iters=50, seed=0):
    """Unsupervised mixture of K linear regressions Y ~ A_k [X,1] (EM). Returns hard labels."""
    rng = np.random.default_rng(seed)
    N, dy = Y.shape
    Xd = np.c_[X, np.ones(N)]
    best, best_ll = None, -np.inf
    for r in range(restarts):
        R = rng.dirichlet(np.ones(K), N)
        for it in range(iters):
            logp = np.zeros((N, K))
            for k in range(K):
                w = R[:, k] + 1e-9
                A = np.linalg.solve(Xd.T @ (Xd * w[:, None]) + 1e-6 * np.eye(Xd.shape[1]), Xd.T @ (Y * w[:, None]))
                E = Y - Xd @ A
                Sig = (E * w[:, None]).T @ E / w.sum() + 1e-4 * np.eye(dy)
                Si = np.linalg.inv(Sig)
                logp[:, k] = (np.log(w.mean()) - 0.5 * np.linalg.slogdet(Sig)[1]
                              - 0.5 * np.einsum("ni,ij,nj->n", E, Si, E))
            mx = logp.max(1, keepdims=True)
            ll = float((mx[:, 0] + np.log(np.exp(logp - mx).sum(1))).sum())
            R = np.exp(logp - mx); R /= R.sum(1, keepdims=True)
        if ll > best_ll:
            best, best_ll = R.argmax(1), ll
    return best


def regime_readouts(Z, Ysame, S, K, shared, n_fit, seed=0):
    """How the regime can be read out of the latents (balanced accuracy, chance = 1/K)."""
    d = Z.shape[-1]
    Zf, Yf, Sf = Z.reshape(-1, d), Ysame.reshape(-1, d), S.reshape(-1)
    sc = StandardScaler().fit(Zf[:n_fit]); Zs = sc.transform(Zf)
    scy = StandardScaler().fit(Yf[:n_fit]); Ys = scy.transform(Yf)
    out = {}
    lin = LogisticRegression(max_iter=2000).fit(Zs[:n_fit], Sf[:n_fit])
    out["probe lin(Z)"] = balanced_accuracy_score(Sf[n_fit:], lin.predict(Zs[n_fit:]))
    mlp = MLPClassifier((64, 64), max_iter=200, early_stopping=True, random_state=seed).fit(Zs[:n_fit], Sf[:n_fit])
    out["probe MLP(Z)"] = balanced_accuracy_score(Sf[n_fit:], mlp.predict(Zs[n_fit:]))
    gm = GaussianMixture(K, n_init=3, random_state=seed).fit(Zs[:n_fit])
    lab = gm.predict(Zs[n_fit:])
    out["GMM(Z) acc"] = best_perm_acc(Sf[n_fit:], lab, K)
    if not shared:
        mlp = MLPClassifier((64, 64), max_iter=200, early_stopping=True, random_state=seed).fit(Ys[:n_fit], Sf[:n_fit])
        out["probe MLP(Y)"] = balanced_accuracy_score(Sf[n_fit:], mlp.predict(Ys[n_fit:]))
        ZY = np.c_[Zs, Ys]
        mlp = MLPClassifier((64, 64), max_iter=200, early_stopping=True, random_state=seed).fit(ZY[:n_fit], Sf[:n_fit])
        out["probe MLP(Z,Y)"] = balanced_accuracy_score(Sf[n_fit:], mlp.predict(ZY[n_fit:]))
        lab = switching_regression(Zs[n_fit:], Ys[n_fit:], K, seed=seed)
        out["switch-reg(Z→Y)"] = best_perm_acc(Sf[n_fit:], lab, K)
        if d == 1 and K == 2:
            out["sign(z·y)"] = best_perm_acc(Sf[n_fit:], (Zs[n_fit:, 0] * Ys[n_fit:, 0] > 0).astype(int), 2)
    return out


def evaluate_latents(Z, Y, spec, lat, shared, n_eval=8000, seed=0, clusters=True):
    """Z, Y: (N,T,d) context / target-encoder outputs on the SAME windows."""
    score, sv = predictable_variance(Z, Y, shared)
    feats = gt_features(spec, lat)
    N, T, d = Z.shape
    take = min(N * T, n_eval)
    idx = np.random.default_rng(seed).permutation(N * T)[:take]
    sub = lambda A: A.reshape(N * T, -1)[idx][None]
    n_fit = int(0.6 * take)
    fsub = {k: sub(v) for k, v in feats.items()}
    res = dict(score=score, canon=np.round(sv, 3))
    res.update({f"R²Z {k}": v for k, v in r2_table(sub(Z), fsub, n_fit).items()})
    if not shared:
        res.update({f"R²Y {k}": v for k, v in r2_table(sub(Y), fsub, n_fit).items()})
    if spec.K > 1:
        res.update(regime_readouts(sub(Z), sub(Y), sub(lat["S"][..., None])[..., 0], spec.K, shared, n_fit, seed))
        if clusters:
            res.update({f"clu {k}": v for k, v in cluster_battery(Z, lat["S"], spec.K, None if shared else Y, seed=seed).items()})
    cov = np.cov(Z.reshape(-1, d).T).reshape(d, d)
    ev = np.linalg.eigvalsh(cov)
    res["latent cov eig (min,max)"] = (round(float(ev.min()), 2), round(float(ev.max()), 2))
    return res

In [ ]:
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA


def _forward_backward(logB, A, pi, lengths):
    """Scaled forward-backward over several sequences. Returns posteriors, expected transition counts, log-lik."""
    N, K = logB.shape
    gamma, xi, ll, start = np.zeros((N, K)), np.zeros((K, K)), 0.0, 0
    for L in lengths:
        lb = logB[start: start + L]
        m = lb.max(1, keepdims=True)
        B = np.exp(lb - m)
        alpha, c = np.zeros((L, K)), np.zeros(L)
        a = pi * B[0]; c[0] = a.sum() + 1e-300; alpha[0] = a / c[0]
        for t in range(1, L):
            a = (alpha[t - 1] @ A) * B[t]; c[t] = a.sum() + 1e-300; alpha[t] = a / c[t]
        beta = np.ones((L, K))
        for t in range(L - 2, -1, -1):
            beta[t] = (A @ (B[t + 1] * beta[t + 1])) / c[t + 1]
        g = alpha * beta
        gamma[start: start + L] = g / g.sum(1, keepdims=True)
        xi += A * (alpha[:-1].T @ (B[1:] * beta[1:] / c[1:, None]))
        ll += np.log(c).sum() + m.sum()
        start += L
    return gamma, xi, ll


def hmm_regression(X, Y, K, lengths, iters=40, restarts=3, seed=0, init=None):
    """HMM whose hidden state k emits Y_t ~ N(A_k [X_t, 1], Σ_k), fitted by Baum-Welch.
    X=None: Gaussian HMM. X=z_t, Y=z_{t+1}: autoregressive HMM (temporal switching regression).
    X=Z, Y=Y on the same input: the switching regression of §6 with a Markov prior on the regime."""
    rng = np.random.default_rng(seed)
    N, dy = Y.shape
    Xd = np.ones((N, 1)) if X is None else np.c_[X, np.ones(N)]
    best, best_ll = None, -np.inf
    for r in range(restarts):
        R = np.eye(K)[init] * 0.8 + 0.2 / K if (r == 0 and init is not None) else rng.dirichlet(np.ones(K), N)
        A = np.full((K, K), 0.1 / max(K - 1, 1)); np.fill_diagonal(A, 0.9)
        pi = np.full(K, 1 / K)
        for it in range(iters):
            logB = np.zeros((N, K))
            for k in range(K):
                w = R[:, k] + 1e-9
                Ak = np.linalg.solve(Xd.T @ (Xd * w[:, None]) + 1e-6 * np.eye(Xd.shape[1]), Xd.T @ (Y * w[:, None]))
                E = Y - Xd @ Ak
                Sig = (E * w[:, None]).T @ E / w.sum() + 1e-4 * np.eye(dy)
                logB[:, k] = -0.5 * np.linalg.slogdet(Sig)[1] - 0.5 * np.einsum("ni,ij,nj->n", E, np.linalg.inv(Sig), E)
            R, xi, ll = _forward_backward(logB, A, pi, lengths)
            A = xi / xi.sum(1, keepdims=True)
        if ll > best_ll:
            best, best_ll = R.argmax(1), ll
    return best


def tsc_cluster(Z, K, q=20, seed=0):
    """Thresholding-based subspace clustering (Heckel & Bölcskei, 2015): each point is linked to its q neighbours in
    absolute cosine similarity, then spectral clustering. Recovers unions of subspaces (regime-specific directions,
    the 'g' features) and is designed for high dimension."""
    Zn = Z - Z.mean(0)
    Zn = Zn / (np.linalg.norm(Zn, axis=1, keepdims=True) + 1e-12)
    C = np.abs(Zn @ Zn.T); np.fill_diagonal(C, 0)
    idx = np.argpartition(-C, q, axis=1)[:, :q]
    rows = np.repeat(np.arange(len(Z)), q)
    vals = np.exp(-2 * np.arccos(np.clip(C[rows, idx.reshape(-1)], 0, 1)))
    from scipy.sparse import csr_matrix
    W = csr_matrix((vals, (rows, idx.reshape(-1))), shape=C.shape)
    W = W + W.T
    return SpectralClustering(K, affinity="precomputed", random_state=seed, assign_labels="cluster_qr").fit_predict(W)


def cluster_battery(Z, S, K, Y=None, n_windows=4000, pcs=8, seed=0):
    """Unsupervised regime recovery from a latent, using contiguous windows (n_series, T, d), so temporal methods apply.
    Every method is scored by balanced accuracy after the best label permutation (chance 1/K)."""
    n_ser = max(1, min(Z.shape[0], int(np.ceil(n_windows / Z.shape[1]))))
    T = min(Z.shape[1], n_windows // n_ser)
    Zc, Sc = Z[:n_ser, :T], S[:n_ser, :T].reshape(-1)
    d = Z.shape[-1]
    flat = StandardScaler().fit_transform(Zc.reshape(-1, d))
    k = min(d, pcs)
    P = PCA(k, whiten=True, random_state=seed).fit_transform(flat) if d > k else flat
    out = {}
    gm = GaussianMixture(K, n_init=3, random_state=seed).fit(P)
    g_lab = gm.predict(P)
    out["GMM"] = best_perm_acc(Sc, g_lab, K)
    out["spectral kNN"] = best_perm_acc(Sc, SpectralClustering(K, affinity="nearest_neighbors", n_neighbors=15,
                                                               random_state=seed, assign_labels="cluster_qr").fit_predict(P), K)
    if d >= 2:
        out["subspace (TSC)"] = best_perm_acc(Sc, tsc_cluster(flat, K, seed=seed), K)
    lengths = [T] * n_ser
    out["HMM"] = best_perm_acc(Sc, hmm_regression(None, P, K, lengths, seed=seed, init=g_lab), K)
    Pm = P.reshape(n_ser, T, -1)
    X_ar, Y_ar = Pm[:, :-1].reshape(-1, k), Pm[:, 1:].reshape(-1, k)
    S_ar = S[:n_ser, :T - 1].reshape(-1)          # S_t governs z_t -> z_{t+1}
    init_ar = g_lab.reshape(n_ser, T)[:, :-1].reshape(-1)
    out["AR-HMM (z_t→z_t+1)"] = best_perm_acc(S_ar, hmm_regression(X_ar, Y_ar, K, [T - 1] * n_ser, seed=seed, init=init_ar), K)
    if Y is not None:
        Yf = StandardScaler().fit_transform(Y[:n_ser, :T].reshape(-1, d))
        Q = PCA(k, whiten=True, random_state=seed).fit_transform(Yf) if d > k else Yf
        out["HMM switch-reg (Z→Y)"] = best_perm_acc(Sc, hmm_regression(P, Q, K, lengths, seed=seed), K)
    return out

In [ ]:
def oracle_dictionary(spec, lat, deg=3):
    """Phi_oracle: all monomials (degree <= deg) of the true state and nuisance factors, each also multiplied by every
    regime indicator, plus the indicators. A linear encoder on this Phi is the setting of the proposal."""
    S, x, n = lat["S"], lat["x"], lat["n"]
    v = np.concatenate([x, n], -1)
    nv = v.shape[-1]
    monos = []
    for dgr in range(1, deg + 1):
        for comb in itertools.combinations_with_replacement(range(nv), dgr):
            monos.append(np.prod(v[..., list(comb)], -1))
    monos = np.stack(monos, -1)
    onehot = np.eye(spec.K)[S]
    parts = [onehot[..., : spec.K - 1]] + [monos * onehot[..., k: k + 1] for k in range(spec.K)]
    return np.concatenate(parts, -1)


def oracle_K(Phi):
    N, T, Fd = Phi.shape
    flat = Phi.reshape(-1, Fd)
    mu = flat.mean(0)
    C = (flat - mu).T @ (flat - mu) / len(flat)
    w, V = np.linalg.eigh(C)
    keep = w > 1e-7 * w.max()
    Wh = V[:, keep] / np.sqrt(w[keep])
    Pw = (Phi - mu) @ Wh
    a, b = Pw[:, :-1].reshape(-1, Wh.shape[1]), Pw[:, 1:].reshape(-1, Wh.shape[1])
    return b.T @ a / len(a), Pw


def shared_optimum(K, d, iters=400, lr=0.2, seed=0):
    """max_U ||U^T K U||_F^2 on the Stiefel manifold (Riemannian ascent, several starts)."""
    rng = np.random.default_rng(seed)
    Ks = (K + K.T) / 2
    w, V = np.linalg.eigh(Ks)
    starts = [V[:, np.argsort(-abs(w))[:d]]]
    Uk, sk, Vkt = np.linalg.svd(K)
    starts += [Uk[:, :d], Vkt[:d].T, np.linalg.qr(Uk[:, :d] + Vkt[:d].T)[0]]
    starts += [np.linalg.qr(rng.standard_normal((K.shape[0], d)))[0] for _ in range(4)]
    best, bestU = -1, None
    for U in starts:
        for _ in range(iters):
            C = U.T @ K @ U
            G = 2 * (K @ U @ C.T + K.T @ U @ C)
            G = G - U @ (U.T @ G + G.T @ U) / 2
            U, _ = np.linalg.qr(U + lr * G)
        f = float(((U.T @ K @ U) ** 2).sum())
        if f > best:
            best, bestU = f, U
    lower = float(np.sum(np.sort(w ** 2)[::-1][:d]))
    upper = float(np.sum(sk[:d] ** 2))
    return best, bestU, lower, upper


def oracle_analysis(spec, d_list, n_series=40, T=2500, seed=123, deg=3, readouts=True):
    """Theory prediction: optimal linear encoders on Phi_oracle (exact whitening, population ~ long simulation)."""
    rng = np.random.default_rng(seed)
    lat = simulate_latents(spec, n_series, T, rng)
    Phi = oracle_dictionary(spec, lat, deg)
    K, Pw = oracle_K(Phi)
    Uk, sk, Vkt = np.linalg.svd(K)
    rows = []
    for d in d_list:
        f_sh, U, lo, up = shared_optimum(K, d)
        for design in ["shared", "separate"]:
            if design == "shared":
                Zc = Yc = Pw @ U
                row = dict(design=design, d=d, score=f_sh, bracket=(round(lo, 3), round(up, 3)))
            else:
                Zc, Yc = Pw @ Vkt[:d].T, Pw @ Uk[:, :d]
                row = dict(design=design, d=d, score=float(np.sum(sk[:d] ** 2)))
            if readouts:
                ev = evaluate_latents(Zc, Yc, spec, lat, design == "shared", seed=seed, clusters=False)
                ev.pop("score"); ev.pop("latent cov eig (min,max)")
                row.update(ev)
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def train_encode(data, design, d, **kw):
    """Train one model and return its context / target latents on every test window, plus training info."""
    t0 = time.time()
    model, hist = train_jepa(data, design=design, d=d, **kw)
    Z, Y = encode(model, data["obs_test"])
    info = dict(final_pred_loss=hist[-1][1], final_reg=hist[-1][2], train_secs=round(time.time() - t0, 1))
    return Z.astype(np.float32), Y.astype(np.float32), info

## 5. Experiments

### E1 — a rotation: what each design can reach (C2, §5)
A 2-D state turns by a quarter turn per step with damping $r=0.9$; no regimes, no cue. The proposal: with the plain
state as features a shared encoder explains $0$ at $d=1$, separate encoders $r^2=0.81$; at $d=2$ both $2r^2=1.62$.
A **learnt** Φ also contains polynomials: $|x|^2$ (and $x_1^2-x_2^2$) predict themselves with correlation $r^2$, so
with a rich Φ the shared optimum at $d=1$ is $r^4\approx0.66$ — still below separate. Both theory rows are shown.

In [ ]:
spec_rot = DataSpec(name="rotation", P=((1.0,),), rho=(0.9,), Q=(R90,), m=2, cue=0.0)
th1 = oracle(spec_rot, [1, 2], deg=3, readouts=False).assign(source="theory, Φ = polynomials ≤ 3")
th1b = oracle(spec_rot, [1, 2], deg=1, readouts=False).assign(source="theory, Φ = linear state")
rows = [cached_run(spec_rot, des, d, tag="E1", seed=0) for d in [1, 2] for des in ["shared", "separate"]]
e1 = pd.DataFrame(rows).assign(source="learnt TCN + SIGReg")
display(pd.concat([th1b, th1, e1])[["source", "design", "d", "score", "canon", "R²Z x", "R²Z |x|²",
                                    "R²Z x quad. (x1²-x2², x1x2)", "latent cov eig (min,max)"]].round(3))
g = lambda df, des, d: float(df[(df.design == des) & (df.d == d)].score.iloc[0])
check("C2 (§5)", "separate > shared in predictable variance at d=1 for a rotation",
      g(e1, "separate", 1) > g(e1, "shared", 1) + 0.05,
      f"separate {g(e1,'separate',1):.3f} vs shared {g(e1,'shared',1):.3f}; theory 0.81 vs 0 (linear Φ) / 0.66 (polynomial Φ)")
check("C2 (§5)", "at d=2 the two designs explain about the same (2r² = 1.62)",
      abs(g(e1, "separate", 2) - g(e1, "shared", 2)) < 0.15,
      f"separate {g(e1,'separate',2):.3f} vs shared {g(e1,'shared',2):.3f}")
check("C2 (§5)", "a shared encoder at d=1 can only use a self-predicting nonlinear feature (|x|² or x1²−x2²), not x",
      e1[(e1.design == "shared") & (e1.d == 1)]["R²Z x"].iloc[0] < 0.2,
      f"R² of x from shared d=1 latent = {e1[(e1.design=='shared')&(e1.d==1)]['R²Z x'].iloc[0]:.2f}")

### E2 — the worked example, sweeping regime persistence $p$ (C1, C3, C4, C5)
Two equally likely regimes, $x_{t+1}=\pm\rho x_t+$noise with $\rho=0.8$, each regime continuing with probability $p$.
Candidate features and their correlation with their own next value (§6–7):

| feature | shared | separate |
|---|---|---|
| regime indicator (mean separation) | $2p-1$ | $2p-1$ |
| $x^2-1$ (regime-free) | $\rho^2=0.64$ | $\rho^2$ |
| one regime direction $g_k$ | $p\rho$ | — |
| pair (context $B_Sx$, target $x$) | — | $\rho=0.8$ for every $p$ |

So at $d=1$ the theory predicts: **shared** keeps $x^2$ for $p\lesssim0.82$ (regime discarded) and the indicator above;
**separate** keeps the regime-dynamics pair (regime only in $\operatorname{sign}(z\cdot y)$) until $2p-1>\rho$, then the indicator.
At $d=3$ there is room for several of them. Two seeds are averaged.

In [ ]:
P_LIST = [0.5, 0.7, 0.9, 0.97]
th2, rows = [], []
for p in P_LIST:
    spec = DataSpec(name=f"worked p={p}", P=two(p))
    th2.append(oracle(spec, [1, 3], deg=3).assign(p=p))
    for d in [1, 3]:
        for des in ["shared", "separate"]:
            for sd in SEEDS:
                rows.append(dict(p=p, **cached_run(spec, des, d, tag="E2", seed=sd)))
th2 = pd.concat(th2).assign(source="theory")
e2_all = pd.DataFrame(rows)
e2 = mean_over_seeds(e2_all, ["p", "design", "d"]).assign(source="learnt")
cols = ["source", "p", "design", "d", "score", "R²Z regime", "R²Z g (x·1[S=k])", "R²Z x", "R²Z |x|²",
        "probe MLP(Z)", "GMM(Z) acc", "probe MLP(Y)", "probe MLP(Z,Y)", "sign(z·y)", "switch-reg(Z→Y)"] + CLU
for d in [1, 3]:
    print(f"--- d = {d} ---")
    show(pd.concat([th2[th2.d == d], e2[e2.d == d]]).sort_values(["p", "design", "source"]), cols)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4))
for des, c in [("shared", C_SH), ("separate", C_SEP)]:
    th, le = th2[(th2.design == des) & (th2.d == 1)], e2[(e2.design == des) & (e2.d == 1)]
    axs[0].plot(th.p, th.score, "--", color=c, alpha=.7); axs[0].plot(le.p, le.score, "o-", color=c, label=des)
    axs[1].plot(le.p, le["probe MLP(Z)"], "o-", color=c, label=f"{des}: MLP probe on Z")
    axs[2].plot(le.p, le["R²Z |x|²"], "o-", color=c, label=f"{des}: R²(|x|²)")
    axs[2].plot(le.p, le["R²Z regime"], "s:", color=c, label=f"{des}: R²(regime indicator)")
le = e2[(e2.design == "separate") & (e2.d == 1)]
axs[1].plot(le.p, le["sign(z·y)"], "^--", color=C_SEP, label="separate: sign(z·y), z,y on same input")
axs[0].set_title("predictable variance, d=1 (dashed = theory)"); axs[1].set_title("regime recovery (chance 0.5)")
axs[2].set_title("which feature the d=1 latent keeps")
for a in axs:
    a.set_xlabel("regime persistence p"); a.legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
q = lambda df, p, des, d, col: float(df[(df.p == p) & (df.design == des) & (df.d == d)][col].iloc[0])
check("C1 (§4,§7)", "p=0.5, d=1: shared encoder keeps |x|² (ρ²=0.64 beats pρ=0.4) and discards the regime",
      q(e2, .5, "shared", 1, "R²Z |x|²") > 0.5 and q(e2, .5, "shared", 1, "probe MLP(Z)") < 0.6,
      f"R²(|x|²)={q(e2,.5,'shared',1,'R²Z |x|²'):.2f}, MLP probe={q(e2,.5,'shared',1,'probe MLP(Z)'):.2f}")
check("C3 (§6)", "p=0.5, d=1: separate encoders beat shared in predictable variance (theory 0.64 vs 0.41)",
      q(e2, .5, "separate", 1, "score") > q(e2, .5, "shared", 1, "score") + 0.05,
      f"{q(e2,.5,'separate',1,'score'):.3f} vs {q(e2,.5,'shared',1,'score'):.3f}")
check("C4 (§6)", "p=0.5, d=1 separate: regime in neither latent alone (MLP probes ≈ 0.5)",
      max(q(e2, .5, "separate", 1, "probe MLP(Z)"), q(e2, .5, "separate", 1, "probe MLP(Y)")) < 0.62,
      f"MLP(Z)={q(e2,.5,'separate',1,'probe MLP(Z)'):.2f}, MLP(Y)={q(e2,.5,'separate',1,'probe MLP(Y)'):.2f}")
check("C4 (§6)", "p=0.5, d=1 separate: regime = sign(z·y) and switching regression recovers it",
      q(e2, .5, "separate", 1, "sign(z·y)") > 0.8 and q(e2, .5, "separate", 1, "switch-reg(Z→Y)") > 0.8,
      f"sign(z·y)={q(e2,.5,'separate',1,'sign(z·y)'):.2f}, switch-reg={q(e2,.5,'separate',1,'switch-reg(Z→Y)'):.2f}")
sep_scores = e2[(e2.design == "separate") & (e2.d == 1) & (e2.p <= 0.7)].score
check("C3 (§6)", "the separate pair's predictability (ρ²) does not depend on p while p is below the indicator's",
      sep_scores.max() - sep_scores.min() < 0.1, f"separate d=1 score at p=0.5/0.7: {np.round(sep_scores.values,3)}")
check("C5 (§6)", "p=0.97, d=1: both designs switch to the regime indicator (mean separation, 2p−1=0.94)",
      min(q(e2, .97, des, 1, "GMM(Z) acc") for des in ["shared", "separate"]) > 0.85,
      f"GMM acc shared {q(e2,.97,'shared',1,'GMM(Z) acc'):.2f}, separate {q(e2,.97,'separate',1,'GMM(Z) acc'):.2f}")

### E2b — carried versus redrawn state (C3)
If the state is redrawn at a switch, the cross-regime blocks vanish and both designs carry the factor $p_{kk}$
(§6, last paragraph): at $p=0.7$ the theory gives separate $0.64\to0.32$ while shared stays near $0.32$–$0.41$.

In [ ]:
rows, th = [], []
for redraw in [False, True]:
    spec = DataSpec(name=f"worked p=0.7 redraw={redraw}", P=two(0.7), redraw=redraw)
    th.append(oracle(spec, [1], deg=3).assign(redraw=redraw, source="theory"))
    for des in ["shared", "separate"]:
        rows.append(dict(redraw=redraw, source="learnt", **cached_run(spec, des, 1, tag="E2b", seed=0)))
e2b = pd.DataFrame(rows)
show(pd.concat([pd.concat(th), e2b]).sort_values(["redraw", "design", "source"]),
     ["source", "redraw", "design", "d", "score", "R²Z g (x·1[S=k])", "R²Z |x|²", "R²Z regime", "probe MLP(Z)", "sign(z·y)"])
r = lambda rd, des: float(e2b[(e2b.redraw == rd) & (e2b.design == des)].score.iloc[0])
check("C3 (§6)", "redrawing the state at a switch removes the separate design's advantage (factor p_kk returns)",
      (r(False, "separate") - r(False, "shared")) > (r(True, "separate") - r(True, "shared")) + 0.05,
      f"gap carried {r(False,'separate')-r(False,'shared'):.3f} vs redrawn {r(True,'separate')-r(True,'shared'):.3f}")

### E3 — mean separation with three regimes, sticky versus cyclic (C5)
Identical state dynamics in all regimes ($\rho=0.8$); regimes differ only through the cue, so only *mean separation*
is possible. **Sticky**: stay with 0.9 (no preferred order, symmetric $K$) — both designs should keep the indicator
(eigenvalue 0.85 > state's 0.8). **Cyclic**: $1\to2\to3\to1$ with probability 0.9 — the indicator block of $K$ is a
damped 120° rotation (modulus 0.85). At $d=1$ a shared encoder only sees its symmetric part (−0.35) and keeps the
state $x$ (0.8) instead; separate encoders keep the regime pair. At $d=2$ the shared encoder can hold the whole plane.

In [ ]:
P_STICKY = ((.9, .05, .05), (.05, .9, .05), (.05, .05, .9))
P_CYCLIC = ((.1, .9, 0.), (0., .1, .9), (.9, 0., .1))
rows, th = [], []
for name, P in [("sticky", P_STICKY), ("cyclic", P_CYCLIC)]:
    spec = DataSpec(name=f"3 regimes {name}", P=P, rho=(.8, .8, .8))   # identical dynamics: only the cue differs
    th.append(oracle(spec, [1, 2], deg=3).assign(order=name, source="theory"))
    for d in [1, 2]:
        for des in ["shared", "separate"]:
            for sd in SEEDS:
                rows.append(dict(order=name, **cached_run(spec, des, d, tag="E3", seed=sd)))
e3 = mean_over_seeds(pd.DataFrame(rows), ["order", "design", "d"]).assign(source="learnt")
th3 = pd.concat(th)
show(pd.concat([th3, e3]).sort_values(["order", "d", "design", "source"]),
     ["source", "order", "design", "d", "score", "R²Z regime", "R²Z x", "GMM(Z) acc", "probe lin(Z)", "probe MLP(Z)", "R²Y regime"] + CLU)
q3 = lambda o, des, d, col: float(e3[(e3.order == o) & (e3.design == des) & (e3.d == d)][col].iloc[0])
check("C5 (§6)", "sticky regimes (no preferred order): both designs keep the indicator at d=1 (clusters, acc≫1/3)",
      min(q3("sticky", des, 1, "GMM(Z) acc") for des in ["shared", "separate"]) > 0.8,
      f"GMM acc shared {q3('sticky','shared',1,'GMM(Z) acc'):.2f}, separate {q3('sticky','separate',1,'GMM(Z) acc'):.2f}")
check("C5 (§6)", "cyclic order, d=1: separate keeps the regime, shared prefers the state x (theory 0.73 vs 0.64)",
      q3("cyclic", "separate", 1, "probe MLP(Z)") > q3("cyclic", "shared", 1, "probe MLP(Z)") + 0.15,
      f"MLP probe separate {q3('cyclic','separate',1,'probe MLP(Z)'):.2f} vs shared {q3('cyclic','shared',1,'probe MLP(Z)'):.2f}")
check("C5 (§6)", "cyclic order, d=2: shared recovers the regime plane",
      q3("cyclic", "shared", 2, "probe MLP(Z)") > 0.8, f"MLP probe shared d=2 {q3('cyclic','shared',2,'probe MLP(Z)'):.2f}")

### E4 — competition for the $d$ dimensions (C6)
No regime dynamics ($\rho=0$), regime predictability $\theta=2p-1=0.6$ ($p=0.8$), plus $k=2$ regime-free AR(1)
nuisances with $\lambda=0.8$. Hermite polynomials of the nuisances have autocorrelations $\lambda,\lambda^2,\dots$; those
above $\theta$ number $\binom{k+M}{k}-1$ with $M=\lfloor\ln\theta/\ln\lambda\rfloor$. The theory therefore places the
regime only after those features have taken their dimensions. $K$ is symmetric here, so both designs should agree.

In [ ]:
D_LIST = [1, 2, 4, 6, 8, 12]
spec_c = DataSpec(name="competition", P=two(0.8), rho=(0.0, 0.0), q=2, lam=0.8)   # regime θ = 2p−1 = 0.6
th4 = oracle(spec_c, D_LIST, deg=3)
rows = [cached_run(spec_c, des, d, tag="E4", seed=0) for d in D_LIST for des in ["shared", "separate"]]
e4 = pd.DataFrame(rows)
M = int(np.floor(np.log(0.6) / np.log(0.8)))
print(f"θ = 0.6, λ = 0.8, k = 2 → M = {M}, polynomials with autocorrelation ≥ θ: C(k+M,k)−1 = {math.comb(2 + M, 2) - 1}"
      f" → regime predicted from d = {math.comb(2 + M, 2)}")
show(pd.concat([th4.assign(source="theory"), e4.assign(source="learnt")]).sort_values(["design", "d", "source"]),
     ["source", "design", "d", "score", "R²Z regime", "R²Z nuisance n", "R²Z nuisance n²", "GMM(Z) acc", "probe lin(Z)"])
fig, axs = plt.subplots(1, 2, figsize=(10, 3.3))
for des, c in [("shared", C_SH), ("separate", C_SEP)]:
    a, b = th4[th4.design == des], e4[e4.design == des]
    axs[0].plot(a.d, a["R²Z regime"], "--", color=c, alpha=.7); axs[0].plot(b.d, b["R²Z regime"], "o-", color=c, label=des)
    axs[1].plot(b.d, b["R²Z nuisance n²"], "o-", color=c, label=f"{des}: nuisance n²")
    axs[1].plot(b.d, b["R²Z nuisance n"], "s:", color=c, label=f"{des}: nuisance n")
axs[0].axvline(math.comb(2 + M, 2), color=C_TH, lw=1); axs[0].set_title("R² of regime indicator (dashed = theory)")
axs[1].set_title("regime-free competitors kept")
for a in axs:
    a.set_xlabel("latent dimension d"); a.legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()
first = lambda df, des: next((d for d, v in zip(df[df.design == des].d, df[df.design == des]["R²Z regime"]) if v > 0.5), None)
check("C6 (§7)", "regime-free features fill the first dimensions: regime absent at d=1",
      e4[e4.d == 1]["R²Z regime"].max() < 0.3, f"R² regime at d=1: {np.round(e4[e4.d==1]['R²Z regime'].values,2)}")
check("C6 (§7)", f"regime appears only once d exceeds the count of stronger competitors (theory d={math.comb(2+M,2)})",
      all((first(e4, des) or 99) >= 4 for des in ["shared", "separate"]),
      f"first d with R²>0.5: shared {first(e4,'shared')}, separate {first(e4,'separate')}; theory {first(th4,'shared')}")

### E5 — the tests §9 lists as missing

**E5a — nonlinear predictor (C8).** §4: "a nonlinear predictor could apply each regime's rule itself, given a latent
that tells it which regime is active". At $p=0.5$, $d=2$ a shared encoder with a linear predictor should not keep the
regime; with an MLP predictor the latent $(x,\text{regime})$ lets the predictor compute $\pm\rho x$.

In [ ]:
spec05 = DataSpec(name="worked p=0.5", P=two(0.5))
rows = [cached_run(spec05, "shared", 2, tag="shared + linear predictor", seed=0),
        cached_run(spec05, "shared", 2, tag="shared + MLP predictor", seed=0, predictor="mlp"),
        cached_run(spec05, "separate", 2, tag="separate + linear predictor", seed=0)]
e5a = pd.DataFrame(rows)
show(e5a, ["tag", "d", "final_pred_loss", "score", "R²Z regime", "R²Z x", "R²Z g (x·1[S=k])", "R²Z |x|²",
           "probe lin(Z)", "probe MLP(Z)", "switch-reg(Z→Y)"])
check("C8 (§4,§9)", "with a nonlinear predictor a shared encoder keeps the regime as a gate even at p=0.5",
      e5a.iloc[1]["probe MLP(Z)"] > e5a.iloc[0]["probe MLP(Z)"] + 0.1,
      f"MLP probe on Z: MLP-pred {e5a.iloc[1]['probe MLP(Z)']:.2f} vs linear-pred {e5a.iloc[0]['probe MLP(Z)']:.2f}")

**E5b — SIGReg versus exact whitening (C7).** Two datasets: the worked example at $p=0.5$ ($d=1$), where the optimum
is a Gaussian-friendly pair, and sticky two-regime mean separation ($p=0.95$, identical dynamics), where the optimum
at $d=1$ is a **two-valued** regime coordinate that a Gaussian-shaped penalty opposes. SIGReg weights
$\lambda\in\{0.03,0.1,1\}$ are compared with exact batch whitening (the theory's constraint).

In [ ]:
spec_ms = DataSpec(name="2 regimes sticky, identical dynamics", P=two(0.95), rho=(.8, .8))
REGS = [("SIGReg λ=0.03", dict(reg="sigreg", lam=0.03)), ("SIGReg λ=0.1", dict(reg="sigreg", lam=0.1)),
        ("SIGReg λ=1", dict(reg="sigreg", lam=1.0)), ("exact whitening", dict(reg="whiten"))]
rows = []
for name, kw in REGS:
    for des in ["shared", "separate"]:
        rows.append(dict(data="worked p=0.5", regulariser=name, **cached_run(spec05, des, 1, tag="E5b", seed=0, **kw)))
        for d in [1, 4]:
            rows.append(dict(data="mean separation p=0.95", regulariser=name, **cached_run(spec_ms, des, d, tag="E5b", seed=0, **kw)))
e5b = pd.DataFrame(rows)
th5b = pd.concat([oracle(spec05, [1], deg=3).assign(data="worked p=0.5"),
                  oracle(spec_ms, [1, 4], deg=3).assign(data="mean separation p=0.95")]).assign(regulariser="theory")
cols = ["data", "regulariser", "design", "d", "score", "R²Z regime", "R²Z x", "R²Z |x|²", "GMM(Z) acc", "sign(z·y)",
        "final_pred_loss", "latent cov eig (min,max)"]
show(pd.concat([th5b, e5b]).sort_values(["data", "d", "design"], kind="stable"), cols)
q5 = lambda data, reg, des, d, col: float(e5b[(e5b.data == data) & (e5b.regulariser == reg) & (e5b.design == des) & (e5b.d == d)][col].iloc[0])
w5 = [q5("worked p=0.5", "exact whitening", des, 1, "score") for des in ["shared", "separate"]]
s5 = [q5("worked p=0.5", "SIGReg λ=1", des, 1, "score") for des in ["shared", "separate"]]
check("C7 (§8)", "a strong SIGReg weight is not a stand-in for exact whitening: λ=1 loses predictable variance",
      np.mean(w5) > np.mean(s5) + 0.1, f"worked p=0.5, d=1 score (shared, separate): whitening {np.round(w5,2)}, SIGReg λ=1 {np.round(s5,2)}")
gap = lambda d: np.mean([q5("mean separation p=0.95", "exact whitening", des, d, "R²Z regime")
                         - q5("mean separation p=0.95", "SIGReg λ=0.1", des, d, "R²Z regime") for des in ["shared", "separate"]])
check("C7 (§8)", "Gaussian shape opposes a two-valued regime coordinate: whitening keeps it at least as well at d=1",
      gap(1) >= -0.05, f"R² regime, whitening − SIGReg(λ=0.1): d=1 {gap(1):.2f}")
check("C7 (§8)", "…and that opposition weakens at larger d", gap(4) <= max(gap(1), 0) + 0.05,
      f"gap d=1 {gap(1):.2f}, d=4 {gap(4):.2f}")

**E5c — robustness of the regression readout (C4, C8).** The separate design at $p=0.5$, $d=1$ under increasing
observation noise and nonlinear warp: does the regime stay out of each latent and recoverable from the pair, now that
the relation $z=\pm y$ only holds approximately?

In [ ]:
rows = []
for noise, warp in [(0.1, 0.0), (0.3, 1.0), (1.0, 1.0), (0.3, 3.0), (1.5, 3.0)]:
    spec = replace(spec05, name=f"noise={noise} warp={warp}", obs_noise=noise, warp=warp)
    rows.append(dict(noise=noise, warp=warp, **cached_run(spec, "separate", 1, tag="E5c", seed=0)))
e5c = pd.DataFrame(rows)
show(e5c, ["noise", "warp", "score", "probe MLP(Z)", "probe MLP(Y)", "probe MLP(Z,Y)", "sign(z·y)", "switch-reg(Z→Y)"] + CLU)
check("C4/C8 (§9)", "the regression readout degrades gracefully: still > 0.7 at noise 1.0",
      float(e5c[(e5c.noise == 1.0)]["switch-reg(Z→Y)"].iloc[0]) > 0.7,
      f"switch-reg acc by condition: {np.round(e5c['switch-reg(Z→Y)'].values,2)}")

**E5d — backbone dependence (C8).** The headline contrast (E2, $p=0.5$, $d=1$) with a TCN, a GRU and a Transformer.

In [ ]:
rows = []
for bb in ["tcn", "gru", "transformer"]:
    for des in ["shared", "separate"]:
        rows.append(cached_run(spec05, des, 1, tag=bb, seed=0, backbone=bb))
e5d = pd.DataFrame(rows)
show(e5d, ["tag", "design", "score", "R²Z |x|²", "R²Z g (x·1[S=k])", "probe MLP(Z)", "probe MLP(Y)", "sign(z·y)", "secs"])
ok = all(float(e5d[(e5d.tag == bb) & (e5d.design == "separate")].score.iloc[0]) >
         float(e5d[(e5d.tag == bb) & (e5d.design == "shared")].score.iloc[0]) for bb in ["tcn", "gru", "transformer"])
check("C8 (§9)", "separate > shared at p=0.5, d=1 holds for every backbone", ok,
      "; ".join(f"{bb}: {e5d[(e5d.tag==bb)&(e5d.design=='separate')].score.iloc[0]:.2f} vs {e5d[(e5d.tag==bb)&(e5d.design=='shared')].score.iloc[0]:.2f}" for bb in ["tcn","gru","transformer"]))

### E6 — challenging data: slow regime-free features and large $d$ (C1, C5, C6)
The hardest setting: three regimes whose dynamics **conflict** (the 2-D state rotates by 0°, 120° or 240° per step
with damping 0.9, so the regime-pooled dynamics average to zero), a **weak** cue (0.5), strong noise (0.5) and warp (2),
and **four slow regime-free factors** with autocorrelation $\lambda\in\{0.9,0.95,0.98\}$, i.e. slow features in the sense of SFA.
Regime predictability: indicator $\approx0.85$ (sticky $p=0.9$), separate regime pair $\rho=0.9$.
By §7, the Hermite polynomials of the slow factors that are at least as predictable number
4 / 34 / 494 (vs. the indicator), so the regime should appear at $d\approx5$, at $d\gtrsim35$, and not at all for $d\le32$.
The oracle uses polynomials of degree ≤ 3 only, so for slow λ it **undercounts** the competitors; the analytic count is printed.
$d\in\{2,4,8,16,32\}$, both designs.

In [ ]:
def rot(deg):
    a = math.radians(deg)
    return ((math.cos(a), -math.sin(a)), (math.sin(a), math.cos(a)))

LAMS, D_BIG = [0.9, 0.95, 0.98], [2, 4, 8, 16, 32]
P3 = ((.9, .05, .05), (.05, .9, .05), (.05, .05, .9))
def spec_slow(lam):
    # 3 regimes whose dynamics conflict (rotations by 0°, 120°, 240°: their average is 0), weak cue, strong noise and warp,
    # plus 4 slow regime-free factors with autocorrelation lam
    return DataSpec(name=f"slow nuisances λ={lam}", P=P3, rho=(.9, .9, .9), Q=(rot(0), rot(120), rot(240)), m=2,
                    q=4, lam=lam, cue=0.5, obs_noise=0.5, warp=2.0)

def n_ahead(lam, theta, k=4):
    M = int(np.floor(np.log(theta) / np.log(lam)))
    return math.comb(k + M, k) - 1

rows, th = [], []
for lam in LAMS:
    spec = spec_slow(lam)
    th.append(oracle(spec, D_BIG, deg=3).assign(slow_lam=lam))
    for d in D_BIG:
        for des in ["shared", "separate"]:
            rows.append(dict(slow_lam=lam, **cached_run(spec, des, d, tag="E6", seed=0)))
e6, th6 = pd.DataFrame(rows), pd.concat(th)
for lam in LAMS:
    print(f"λ={lam}: nuisance polynomials more predictable than the regime indicator (θ=0.85): {n_ahead(lam, .85)}, "
          f"than the separate regime pair (θ=0.9): {n_ahead(lam, .9)}")
cols = ["source", "slow_lam", "design", "d", "score", "R²Z regime", "R²Z g (x·1[S=k])", "R²Z |x|²", "R²Z nuisance n",
        "R²Z nuisance n²", "probe lin(Z)", "probe MLP(Z)", "GMM(Z) acc", "probe MLP(Z,Y)", "switch-reg(Z→Y)"] + CLU
show(pd.concat([th6.assign(source="theory (Φ deg ≤ 3)"), e6.assign(source="learnt")]).sort_values(["slow_lam", "design", "d", "source"]), cols)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(13, 6.2), sharex=True, sharey="row")
for j, lam in enumerate(LAMS):
    for des, c in [("shared", C_SH), ("separate", C_SEP)]:
        b, a = e6[(e6.slow_lam == lam) & (e6.design == des)], th6[(th6.slow_lam == lam) & (th6.design == des)]
        axs[0, j].plot(b.d, b["probe MLP(Z)"], "o-", color=c, label=f"{des}: MLP probe")
        axs[0, j].plot(b.d, b["GMM(Z) acc"], "s:", color=c, label=f"{des}: GMM (mean sep.)")
        axs[0, j].plot(a.d, a["probe MLP(Z)"], "--", color=c, alpha=.5, lw=1.2, label=f"{des}: theory probe")
        axs[1, j].plot(b.d, b["R²Z nuisance n"], "o-", color=c, label=f"{des}: slow n")
        axs[1, j].plot(b.d, b["R²Z nuisance n²"], "s:", color=c, label=f"{des}: slow n²")
    axs[0, j].axhline(1 / 3, color=C_TH, lw=1); axs[0, j].set_title(f"λ = {lam}: regime recovery from Z (chance 1/3)")
    axs[0, j].axvline(n_ahead(lam, .85) + 1, color=C_TH, lw=1, ls="--")
    axs[1, j].set_title(f"λ = {lam}: slow regime-free features kept (R²)"); axs[1, j].set_xlabel("latent dimension d")
    axs[1, j].set_xscale("log", base=2); axs[1, j].set_xticks(D_BIG, [str(d) for d in D_BIG])
axs[0, 0].legend(frameon=False, fontsize=6.5); axs[1, 0].legend(frameon=False, fontsize=7)
fig.text(0.5, -0.01, "dashed vertical line: §7 count of nuisance polynomials ahead of the regime indicator + 1 (off-axis when > 32)", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
onset = lambda lam, des: next((d for d, v in zip(e6[(e6.slow_lam == lam) & (e6.design == des)].d,
                                                 e6[(e6.slow_lam == lam) & (e6.design == des)]["probe MLP(Z)"]) if v > 0.8), None)
on = {(lam, des): onset(lam, des) for lam in LAMS for des in ["shared", "separate"]}
print("first d with MLP probe > 0.8:", on)
check("C6 (§7)", "slow regime-free features fill d at small d: slow n is kept before the regime at d=2 (λ=0.98)",
      e6[(e6.slow_lam == 0.98) & (e6.d == 2)]["R²Z nuisance n"].min() > 0.3,
      f"R²(slow n) at d=2: {np.round(e6[(e6.slow_lam==0.98)&(e6.d==2)]['R²Z nuisance n'].values,2)}")
key = lambda v: 99 if v is None else v
check("C6 (§7)", "the regime appears at larger d as the competitors get slower (onset non-decreasing in λ)",
      all(key(on[(0.9, des)]) <= key(on[(0.95, des)]) <= key(on[(0.98, des)]) for des in ["shared", "separate"]),
      f"onsets {on}")
check("C6 (§7)", "with λ=0.98 the §7 count (494 features ahead) predicts no regime up to d=32",
      all(on[(0.98, des)] is None for des in ["shared", "separate"]), f"onset λ=0.98: shared {on[(0.98,'shared')]}, separate {on[(0.98,'separate')]}")
check("C6 (§7)", "conflicting dynamics with a slow competitor: separate encoders reach the regime no later than shared",
      all(key(on[(lam, "separate")]) <= key(on[(lam, "shared")]) for lam in LAMS), f"onsets {on}")

### Unsupervised regime recovery in high dimension
A GMM finds regimes only when they form clusters (mean separation). The latents also hold the regime as **subspaces**
(the $g$ features), as **dynamics** (different linear maps from $z_t$ to $z_{t+1}$), or, for separate encoders,
as the **relation between $Z$ and $Y$**. Each readout targets one of these. All are fitted without labels on 4000
contiguous test windows:

| method | finds regimes that differ in | reference |
|---|---|---|
| GMM on PCA-8 | means / covariances (clusters) | – |
| spectral kNN | non-convex clusters, manifold structure | Ng, Jordan & Weiss 2002 |
| subspace (TSC) | the directions they occupy (union of subspaces), robust in high dimension | Heckel & Bölcskei 2015 |
| HMM | means / covariances + regime persistence | – |
| AR-HMM $z_t\to z_{t+1}$ | linear dynamics in the latent (temporal switching regression) + persistence | – |
| HMM switch-reg $Z\to Y$ | the map between context and target latents on the same input (§6) + persistence | Quandt 1958; De Veaux 1989 |

Below: every run whose latent contains the regime (supervised MLP probe ≥ 0.9), and how well each unsupervised
method finds it.

In [ ]:
allruns = pd.DataFrame([dict(v, spec=k[0].split("name='")[1].split("'")[0]) for k, v in _cache.items()
                        if v.get("readout_version") == READOUT_VERSION and "probe MLP(Z)" in v])
present = allruns[allruns["probe MLP(Z)"] >= 0.9]          # runs whose latent does contain the regime
unsup = [c for c in ["GMM(Z) acc", "switch-reg(Z→Y)", "sign(z·y)"] + CLU if c in allruns]
tab = present.groupby("design")[unsup].mean().T
tab["share of runs ≥ 0.9 (shared)"] = [(present[present.design == "shared"][c] >= 0.9).mean() for c in unsup]
tab["share of runs ≥ 0.9 (separate)"] = [(present[present.design == "separate"][c] >= 0.9).mean() for c in unsup]
print(f"{len(present)} of {len(allruns)} runs have the regime in Z (MLP probe ≥ 0.9). Mean unsupervised accuracy on those:")
display(tab.round(3))
fig, ax = plt.subplots(figsize=(11, 0.28 * len(present) + 1.5))
M = present.sort_values(["design", "spec", "d"])
mat = M[["probe MLP(Z)"] + unsup].values.astype(float)
im = ax.imshow(mat, aspect="auto", cmap="Blues", vmin=0.3, vmax=1)
ax.set_xticks(range(mat.shape[1]), ["MLP probe (supervised)"] + [c.replace("clu ", "") for c in unsup], rotation=35, ha="right")
ax.set_yticks(range(len(M)), [f"{r.design[:3]} d={r.d} {r.spec}" for r in M.itertuples()], fontsize=6.5)
ax.grid(False); plt.colorbar(im, ax=ax, label="balanced accuracy"); ax.set_title("Unsupervised regime recovery where the latent contains the regime")
plt.tight_layout(); plt.show()
best_unsup = present[[c for c in CLU if c in present]].max(1)
check("readout", "some unsupervised method recovers the regime (≥ 0.9) whenever a supervised probe does",
      (best_unsup >= 0.9).mean() >= 0.8, f"share of such runs where the best clustering method reaches 0.9: {(best_unsup >= 0.9).mean():.2f}; "
      f"plain GMM: {(present['GMM(Z) acc'] >= 0.9).mean():.2f}")

## 6. Scoreboard

In [ ]:
summary = pd.DataFrame(CHECKS)
with pd.option_context("display.max_colwidth", 200):
    display(summary)
print(f"{(summary.consistent == 'yes').sum()} / {len(summary)} predictions consistent with the learnt models")

## 7. Reading the results, and what this notebook does not show

* The oracle rows are the proposal's theory on a specific polynomial Φ and a finite simulation; the learnt rows add
  everything the theory leaves out (learnt Φ, SIGReg, SGD, noise). Disagreements are informative about *which*
  assumption breaks: compare `latent cov eig` (how far from whitened), `final_pred_loss`, and the R² profile.
* `score` is measured after **post-hoc exact whitening**, so it measures the subspace the encoder found, not the
  value of the training loss under SIGReg.
* Optimisation matters: $C=0$ is a stationary point of $\|C\|_F^2$, and a strong SIGReg weight can hold the latent at
  a Gaussian feature with $C=0$ (see E5b, $\lambda=1$). This is outside the proposal's statements, which describe
  optima, not what training reaches.
* All results are for one generator family, $M_k=I$, two or three regimes and windows of 16 samples; seeds are few.
  The checks use fixed thresholds that are stated in each line.